In [3]:
%pip install sdv pandas numpy plt sns

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sdv.datasets.demo import download_demo
from sdv.metadata import SingleTableMetadata
from sdv.single_table import TVAESynthesizer, GaussianCopulaSynthesizer
from rdt.transformers.numerical import GaussianNormalizer, ClusterBasedNormalizer, FloatFormatter
from scipy.stats import ks_2samp, chi2_contingency

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Cargar el dataset de ejemplo oficial
# Este dataset está pre-limpiado para pruebas de síntesis
url = "../../datasources/bank_full_dataset/bank-full.csv"
df_real = pd.read_csv(url, sep=';')
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_real)

print("Columnas del dataset:", df_real.columns.tolist())
df_real.head()

num_cols = df_real.select_dtypes(include=[np.number]).columns
print("Columnas numericas:", num_cols)
cat_cols = df_real.select_dtypes(exclude=[np.number]).columns
print("Columnas categoricas:", cat_cols)


Columnas del dataset: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Columnas numericas: Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')
Columnas categoricas: Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')


In [5]:
import time

# Configurar el sintetizador por copula

synthesizer = GaussianCopulaSynthesizer(
    metadata,
    enforce_min_max_values=True,
    default_distribution=None # probar 'gamma' si balance sigue fallando
)

# Tratamiento de precisión para pasar el Test K-S
# El GaussianNormalizer es más estricto y fiel para distribuciones continuas
synthesizer.auto_assign_transformers(df_real)

synthesizer.update_transformers(column_name_to_transformer={
    #'balance': ClusterBasedNormalizer(model_missing_values=True),
    'duration': GaussianNormalizer(model_missing_values=True),
    'day': ClusterBasedNormalizer(model_missing_values=True, enforce_min_max_values=True, max_clusters=31),
    'pdays': ClusterBasedNormalizer(model_missing_values=True),
})

print("Entrenando modelo synthesizer = GaussianCopulaSynthesizer...")
start_time = time.perf_counter()

synthesizer.fit(df_real)

end_time = time.perf_counter()
elapsed_time = end_time - start_time


synthesizer.save("gaussian_copula.pkl")

print(f"Entrenamiento terminado en {elapsed_time:.4f} segundos.")



Entrenando modelo synthesizer = GaussianCopulaSynthesizer...
Entrenamiento terminado en 29.8914 segundos.


In [6]:
# Generar datos sintéticos
df_synth = synthesizer.sample(num_rows=len(df_real))
df_synth['age'] = df_synth['age'].round().astype(int)
df_synth['day'] = df_synth['day'].round().astype(int)

df_synth.to_csv('copula_synth_out.csv')


# Función de reporte de calidad
def reporte_calidad(real, synth):
    print("\n" + "="*40)
    print("   REPORTE DE FIDELIDAD ESTADÍSTICA")
    print("="*40 + "\n")
      
    # Test de Proporciones para Categóricas (Diferencia Media)
    cat_cols = real.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        f_real = real[col].value_counts(normalize=True).sort_index()
        f_synth = synth[col].value_counts(normalize=True).sort_index()
        diff = (f_real - f_synth).abs().mean()
        status = "✅ FIEL" if diff < 0.05 else "❌ DESVIADO"
        print(f"Prop. Diff [{col:10}]: {diff:.4f} {status}")

reporte_calidad(df_real, df_synth)


   REPORTE DE FIDELIDAD ESTADÍSTICA

Prop. Diff [job       ]: 0.0010 ✅ FIEL
Prop. Diff [marital   ]: 0.0048 ✅ FIEL
Prop. Diff [education ]: 0.0020 ✅ FIEL
Prop. Diff [default   ]: 0.0004 ✅ FIEL
Prop. Diff [housing   ]: 0.0012 ✅ FIEL
Prop. Diff [loan      ]: 0.0006 ✅ FIEL
Prop. Diff [contact   ]: 0.0004 ✅ FIEL
Prop. Diff [month     ]: 0.0012 ✅ FIEL
Prop. Diff [poutcome  ]: 0.0007 ✅ FIEL
Prop. Diff [y         ]: 0.0007 ✅ FIEL
